# CUDA Optimization Course - Module 2: Automatic Mixed Precision (AMP)

Reduce memory usage and increase throughput using FP16/BF16 precision.

## Setup and Device Detection

In [ ]:
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
import time

# Detect device
if torch.cuda.is_available():
    device = 'cuda'
    print(f"CUDA available: {torch.cuda.get_device_name()}")
    print(f"Supports FP16: {torch.cuda.is_available()}")
elif torch.backends.mps.is_available():
    device = 'mps'
    print("Using MPS (M3 MacBook)")
else:
    device = 'cpu'
    print("Using CPU")

print(f"Device: {device}")

## Understanding Precision and Dtypes

In [ ]:
print("PyTorch Data Types:")
print(f"FP32 (float32): 32-bit - Full precision, standard")
print(f"FP16 (float16): 16-bit - Half precision, ±65,504 range")
print(f"BF16 (bfloat16): 16-bit - Brain float, better range than FP16")
print(f"TF32: Tensor float, efficient on modern GPUs")

# Create tensors in different precisions
x = torch.randn(2, 2)
print(f"\nDefault dtype: {x.dtype}")
print(f"Memory per element (FP32): {x.element_size()} bytes")

x_fp16 = x.half()
print(f"FP16 dtype: {x_fp16.dtype}")
print(f"Memory per element (FP16): {x_fp16.element_size()} bytes")
print(f"Memory savings: {(1 - x_fp16.element_size() / x.element_size()) * 100:.0f}%")

## Transformer Model for AMP Testing

In [ ]:
class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size=10000, d_model=512, nhead=8, num_layers=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=2048,
                batch_first=True,
                dropout=0.1
            ),
            num_layers=num_layers
        )
        self.fc = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        x = self.fc(x)
        return x

model = SimpleTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
print(f"Model loaded on {device}")

## FP32 Baseline Training (Full Precision)

In [ ]:
batch_size = 32
seq_length = 256
num_iterations = 50

def create_batch():
    input_ids = torch.randint(0, 10000, (batch_size, seq_length)).to(device)
    target_ids = torch.randint(0, 10000, (batch_size, seq_length)).to(device)
    return input_ids, target_ids

# Warmup
input_ids, target_ids = create_batch()
output = model(input_ids)
loss = F.cross_entropy(output.view(-1, 10000), target_ids.view(-1))
loss.backward()
optimizer.zero_grad()

# Measure FP32
torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
start = time.perf_counter()

for _ in range(num_iterations):
    optimizer.zero_grad()
    input_ids, target_ids = create_batch()
    output = model(input_ids)
    loss = F.cross_entropy(output.view(-1, 10000), target_ids.view(-1))
    loss.backward()
    optimizer.step()

torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
fp32_time = time.perf_counter() - start

print(f"FP32 Training (50 iterations):")
print(f"Time: {fp32_time:.2f}s")
print(f"Throughput: {num_iterations * batch_size / fp32_time:.0f} samples/sec")

## FP16/Mixed Precision Training with AMP

In [ ]:
# Reset model
model = SimpleTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# Create GradScaler for automatic loss scaling
scaler = GradScaler(device=device)

# Warmup with AMP
input_ids, target_ids = create_batch()
with autocast(device_type=device, dtype=torch.float16):
    output = model(input_ids)
    loss = F.cross_entropy(output.view(-1, 10000), target_ids.view(-1))
loss.backward()
optimizer.zero_grad()

# Measure FP16 with AMP
torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
start = time.perf_counter()

for _ in range(num_iterations):
    optimizer.zero_grad()
    input_ids, target_ids = create_batch()
    
    # Autocast enables automatic precision selection
    with autocast(device_type=device, dtype=torch.float16):
        output = model(input_ids)
        loss = F.cross_entropy(output.view(-1, 10000), target_ids.view(-1))
    
    # Scale loss and backward (automatic loss scaling)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
amp_time = time.perf_counter() - start

print(f"FP16 AMP Training (50 iterations):")
print(f"Time: {amp_time:.2f}s")
print(f"Throughput: {num_iterations * batch_size / amp_time:.0f} samples/sec")
print(f"\nSpeedup: {fp32_time / amp_time:.2f}x")

## AMP with Gradient Accumulation

In [ ]:
model = SimpleTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = GradScaler(device=device)

accumulation_steps = 4
total_batches = num_iterations * accumulation_steps

torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
start = time.perf_counter()

for i in range(total_batches):
    input_ids, target_ids = create_batch()
    
    with autocast(device_type=device, dtype=torch.float16):
        output = model(input_ids)
        loss = F.cross_entropy(output.view(-1, 10000), target_ids.view(-1))
        loss = loss / accumulation_steps  # Normalize loss
    
    scaler.scale(loss).backward()
    
    if (i + 1) % accumulation_steps == 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

torch.mps.synchronize() if device == 'mps' else torch.cuda.synchronize() if device == 'cuda' else None
accum_time = time.perf_counter() - start

print(f"FP16 AMP with Gradient Accumulation (4x):")
print(f"Time: {accum_time:.2f}s")
print(f"Throughput: {total_batches * batch_size / accum_time:.0f} samples/sec")

## AMP Context Manager Usage Patterns

In [ ]:

# Pattern 1: Simple forward pass with AMP
with autocast(device_type='cuda', dtype=torch.float16):
    output = model(input)
    loss = criterion(output, target)

# Pattern 2: Mixed precision training loop
scaler = GradScaler(device='cuda')

for input, target in dataloader:
    optimizer.zero_grad()
    
    with autocast(device_type='cuda', dtype=torch.float16):
        output = model(input)
        loss = criterion(output, target)
    
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

# Pattern 3: Exclude layers from AMP
with autocast(device_type='cuda', dtype=torch.float16):
    output = model(input)
    with autocast(dtype=torch.float32):  # Force FP32
        loss = criterion(output, target)  # Use full precision


## Key Takeaways

1. **AMP Benefits**: 2-3x faster training with FP16, ~50% memory savings
2. **GradScaler**: Prevents gradient underflow by scaling losses
3. **Autocast**: Automatically selects precision for each operation
4. **Stability**: Mix FP16 (compute) with FP32 (numerical operations)
5. **Performance**: Works best with large models and batch sizes